# UTD-MHAD: Comprehensive Per-Fold Feature Selection Comparison

This notebook compares **17 feature selection methods** across all 8 LOSO folds (treated individually):

**Standard Methods:**
1. Mutual Information (filter)
2. RFE (wrapper)
3. LASSO/L1 (embedded)

**14 EvoloPy Metaheuristics:**
BAT, CS, DE, FFA, GA, GWO, HHO, JAYA, MFO, MVO, PSO, SCA, SSA, WOA


In [2]:
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import time
import warnings
from pathlib import Path
from tqdm import tqdm
from copy import deepcopy
import random
import sys
from scipy import stats
import psutil
import tracemalloc
import gc
import json as json_lib

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset

from sklearn.model_selection import GroupKFold
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA, KernelPCA, SparsePCA
try:
    import umap
except ImportError:
    umap = None  # install with: pip install umap-learn
from sklearn.metrics import (accuracy_score, confusion_matrix, f1_score,
                             precision_score, recall_score, classification_report)

# Standard feature selection imports
from sklearn.feature_selection import mutual_info_classif, RFE, SelectKBest
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression

# Import ALL EvoloPy optimizers
sys.path.append('../EvoloPy-master')
from EvoloPy.optimizers import BAT, CS, DE, FFA, GA, GWO, HHO, JAYA, MFO, MVO, PSO, SCA, SSA, WOA

warnings.filterwarnings('ignore')
np.random.seed(42)
torch.manual_seed(42)
random.seed(42)

# Global device
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")
if torch.cuda.is_available():
    print(f"PyTorch version: {torch.__version__}")
    print(f"GPU: {torch.cuda.get_device_name(0)}")


c:\Users\aruay.amangeldi\Desktop\mhar-feature-selection\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Device: cuda
PyTorch version: 2.11.0+cu126
GPU: NVIDIA GeForce RTX 4060 Laptop GPU


## Configuration

In [3]:
# Paths
FEATURE_DIR = Path("features")

# Master output directory
RESULTS_ROOT = Path("results_depth_skeleton")
RESULTS_ROOT.mkdir(exist_ok=True)
PLOTS_DIR = RESULTS_ROOT / "plots"
PLOTS_DIR.mkdir(exist_ok=True)

# Hyperparameters
VAL_SUBJECT = 8  # Subject index (1-based) dedicated for validation
N_EPOCHS = 300
BATCH_SIZE = 32
LEARNING_RATE = 0.001
HIDDEN_DIM = 128

DEPTH_PCA_DIM = 512

DEPTH_DR_METHOD = 'kernel_pca'

# Feature selection hyperparameters - metaheuristic
N_POPULATION = 20
MAX_ITERATIONS = 30

# For standard methods (target: select ~50% of features)
TARGET_FEATURE_PERCENTAGE = 0.5

# All EvoloPy optimizer names and their callable entry points
EVOLOPY_OPTIMIZERS = {
    'BAT': BAT.BAT,
    'CS':  CS.CS,
    'DE':  DE.DE,
    'FFA': FFA.FFA,
    'GA':  GA.GA,
    'GWO': GWO.GWO,
    'HHO': HHO.HHO,
    'JAYA': JAYA.JAYA,
    'MFO': MFO.MFO,
    'MVO': MVO.MVO,
    'PSO': PSO.PSO,
    'SCA': SCA.SCA,
    'SSA': SSA.SSA,
    'WOA': WOA.WOA,
}

ALL_METHODS = ['baseline', 'mutual_info', 'rfe', 'lasso'] + [f'meta_{k}' for k in EVOLOPY_OPTIMIZERS.keys()]

# Create per-method subdirectories
for method in ALL_METHODS:
    (RESULTS_ROOT / method).mkdir(exist_ok=True)

print(f"Configuration:")
print(f"  VAL_SUBJECT: {VAL_SUBJECT} (dedicated validation subject)")
print(f"  N_EPOCHS: {N_EPOCHS}")
print(f"  Target feature selection: {TARGET_FEATURE_PERCENTAGE*100:.0f}%")
print(f"  Metaheuristic pop: {N_POPULATION}, iter: {MAX_ITERATIONS}")
print(f"  EvoloPy optimizers: {list(EVOLOPY_OPTIMIZERS.keys())}")
print(f"  Total methods (incl. baseline): {len(ALL_METHODS)}")


Configuration:
  VAL_SUBJECT: 8 (dedicated validation subject)
  N_EPOCHS: 300
  Target feature selection: 50%
  Metaheuristic pop: 20, iter: 30
  EvoloPy optimizers: ['BAT', 'CS', 'DE', 'FFA', 'GA', 'GWO', 'HHO', 'JAYA', 'MFO', 'MVO', 'PSO', 'SCA', 'SSA', 'WOA']
  Total methods (incl. baseline): 18


## Load Data

In [4]:
# Load features
X_feat = joblib.load(FEATURE_DIR / "X_feat.pkl")
y = np.load(FEATURE_DIR / "y.npy")
subjects = np.load(FEATURE_DIR / "subjects.npy")
le = joblib.load(FEATURE_DIR / "label_encoder.pkl")
print(f"Loaded {len(X_feat)} samples")
print(f"Number of classes: {len(np.unique(y))}")
print(f"Number of subjects: {len(np.unique(subjects))}")

# Get feature dimensions from first sample
first_sample = X_feat[0]
MODALITY_KEYS = ['depth_feat', 'skeleton_feat']
MODALITY_NAMES = ['depth', 'skeleton']

RAW_FEATURE_DIMS = {}
for key, name in zip(MODALITY_KEYS, MODALITY_NAMES):
    RAW_FEATURE_DIMS[name] = first_sample[key].shape[0]

# Store per-modality arrays (N_samples x D_modality)
X_per_modality = {}
for key, name in zip(MODALITY_KEYS, MODALITY_NAMES):
    X_per_modality[name] = np.array([s[key] for s in X_feat])

print(f"\nRaw feature dimensions (before PCA):")
for name, dim in RAW_FEATURE_DIMS.items():
    print(f"  {name}: {dim}")
print(f"  TOTAL: {sum(RAW_FEATURE_DIMS.values())}")

# FEATURE_DIMS will be set dynamically per-fold after PCA
# (depth dim changes if PCA is applied)
print(f"\nDepth PCA target: {DEPTH_PCA_DIM}" if DEPTH_PCA_DIM else "\nDepth PCA: disabled")


Loaded 861 samples
Number of classes: 27
Number of subjects: 8

Raw feature dimensions (before PCA):
  depth: 5508
  skeleton: 1879
  TOTAL: 7387

Depth PCA target: 512


## Neural Network Model (Same as Original)

In [5]:
class MultiModalDataset(Dataset):
    def __init__(self, features, labels):
        self.features = torch.FloatTensor(features)
        self.labels = torch.LongTensor(labels)
    
    def __len__(self):
        return len(self.labels)
    
    def __getitem__(self, idx):
        return self.features[idx], self.labels[idx]


class SimpleNN(nn.Module):
    """MLP for unified feature vector (adaptive to feature subset size)"""
    def __init__(self, input_dim, num_classes):
        super().__init__()
        
        # Adaptive hidden layer sizing based on input dimension
        hidden1 = max(128, min(512, input_dim * 2))
        hidden2 = max(64, min(256, hidden1 // 2))
        
        self.classifier = nn.Sequential(
            nn.Linear(input_dim, hidden1),
            nn.BatchNorm1d(hidden1),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(hidden1, hidden2),
            nn.BatchNorm1d(hidden2),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(hidden2, num_classes)
        )
    
    def forward(self, x):
        return self.classifier(x)

print("Neural network model defined")


Neural network model defined


## Helper Functions

In [6]:
def prepare_fold_data_per_modality(X_per_modality, train_idx, val_idx, test_idx,
                                    depth_pca_dim=None):
    """
    Per-modality pipeline: split -> normalize per modality -> PCA on depth -> concatenate.
    
    Returns:
        X_train, X_val, X_test: normalized (and PCA-reduced) concatenated arrays
        feature_dims: dict of {modality: dim} AFTER PCA (needed for retention calc)
        pca_obj: fitted PCA object (or None) for reference
    """
    modality_train = {}
    modality_val = {}
    modality_test = {}
    scalers = {}
    pca_obj = None
    feature_dims = {}
    
    for name in MODALITY_NAMES:
        X_mod = X_per_modality[name]
        
        # Split
        X_tr = X_mod[train_idx]
        X_v  = X_mod[val_idx]
        X_te = X_mod[test_idx]
        
        # Per-modality normalization (fit on train only)
        scaler = StandardScaler()
        X_tr = scaler.fit_transform(X_tr)
        X_v  = scaler.transform(X_v)
        X_te = scaler.transform(X_te)
        scalers[name] = scaler
        
        # Dimensionality reduction on depth only (method chosen by DEPTH_DR_METHOD)
        if name == 'depth' and depth_pca_dim is not None and depth_pca_dim < X_tr.shape[1]:
            dr_method = DEPTH_DR_METHOD.lower()

            if dr_method == 'kernel_pca':
                # ── Kernel PCA (non-linear, rbf kernel) ────────────────────
                pca_obj = KernelPCA(n_components=depth_pca_dim, kernel='rbf',
                                    random_state=42, n_jobs=-1)
                X_tr = pca_obj.fit_transform(X_tr)
                X_v  = pca_obj.transform(X_v)
                X_te = pca_obj.transform(X_te)
                print(f"    KernelPCA on depth: {RAW_FEATURE_DIMS['depth']} -> {depth_pca_dim}")

            else:
                raise ValueError(f"Unknown DEPTH_DR_METHOD: '{DEPTH_DR_METHOD}'. ")
        
        modality_train[name] = X_tr
        modality_val[name]   = X_v
        modality_test[name]  = X_te
        feature_dims[name]   = X_tr.shape[1]
    
    # Concatenate: depth | skeleton
    X_train = np.concatenate([modality_train[n] for n in MODALITY_NAMES], axis=1)
    X_val   = np.concatenate([modality_val[n]   for n in MODALITY_NAMES], axis=1)
    X_test  = np.concatenate([modality_test[n]  for n in MODALITY_NAMES], axis=1)
    
    total = sum(feature_dims.values())
    print(f"    Feature dims after processing: " + 
          " | ".join(f"{n}={feature_dims[n]}" for n in MODALITY_NAMES) +
          f" | TOTAL={total}")
    
    return X_train, X_val, X_test, feature_dims, pca_obj


def prepare_unified_features(X_feat_list, feature_mask=None):
    """Concatenate all modality features into unified vector (legacy, used for masks)"""
    unified_features = []
    
    for sample in X_feat_list:
        feat_vector = np.concatenate([
            sample['depth_feat'],
            sample['skeleton_feat']
        ])
        
        if feature_mask is not None:
            feat_vector = feat_vector[feature_mask]
        
        unified_features.append(feat_vector)
    
    return np.array(unified_features)

def calculate_modality_retention(binary_mask, feature_dims):
    """Calculate how many features retained per modality"""
    start_idx = 0
    retention = {}
    
    for modality in MODALITY_NAMES:
        dim = feature_dims[modality]
        end_idx = start_idx + dim
        modality_mask = binary_mask[start_idx:end_idx]
        num_selected = np.sum(modality_mask)
        percentage = (num_selected / dim) * 100
        
        retention[modality] = {
            'selected': int(num_selected),
            'total': dim,
            'percentage': percentage
        }
        start_idx = end_idx
    
    return retention

def train_and_evaluate(model, train_loader, val_loader, test_loader, num_epochs, lr):
    """Train model and return metrics"""
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    
    model.train()
    for epoch in range(num_epochs):
        for features, labels in train_loader:
            features = features.to(DEVICE)
            labels = labels.to(DEVICE)
            
            optimizer.zero_grad()
            outputs = model(features)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
    
    # Evaluate
    model.eval()
    with torch.no_grad():
        # Validation
        val_preds, val_true = [], []
        for features, labels in val_loader:
            features = features.to(DEVICE)
            outputs = model(features)
            preds = torch.argmax(outputs, dim=1).cpu().numpy()
            val_preds.extend(preds)
            val_true.extend(labels.numpy())
        val_acc = accuracy_score(val_true, val_preds)
        
        # Test
        test_preds, test_true = [], []
        for features, labels in test_loader:
            features = features.to(DEVICE)
            outputs = model(features)
            preds = torch.argmax(outputs, dim=1).cpu().numpy()
            test_preds.extend(preds)
            test_true.extend(labels.numpy())
        test_acc = accuracy_score(test_true, test_preds)
    
    return val_acc, test_acc

def count_model_parameters(model):
    """Count trainable parameters in a model"""
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

def get_model_size_mb(model):
    """Get model size in MB"""
    param_size = sum(p.nelement() * p.element_size() for p in model.parameters())
    buffer_size = sum(b.nelement() * b.element_size() for b in model.buffers())
    return (param_size + buffer_size) / (1024 ** 2)

def get_gpu_memory_mb():
    """Get current GPU memory usage in MB"""
    if torch.cuda.is_available():
        return torch.cuda.memory_allocated() / (1024 ** 2)
    return 0.0

def get_dataset_size_mb(X):
    """Get dataset size in MB"""
    return X.nbytes / (1024 ** 2)

print("Helper functions defined (with per-modality normalization + depth PCA)")


Helper functions defined (with per-modality normalization + depth PCA)


## Enhanced Evaluation (returns full metrics per fold)

In [7]:
def train_and_evaluate_full(model, train_loader, val_loader, test_loader, num_epochs, lr, num_classes):
    """Train model and return comprehensive metrics including predictions for confusion matrix"""
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    
    # Track GPU memory before training
    gpu_mem_before = get_gpu_memory_mb()
    train_start = time.time()

    train_losses = []
    best_val_acc = -1.0
    best_model_state = None
    
    for epoch in range(num_epochs):
        model.train()
        epoch_loss = 0.0

        for features, labels in train_loader:
            features = features.to(DEVICE)
            labels = labels.to(DEVICE)
            optimizer.zero_grad()
            outputs = model(features)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            epoch_loss += loss.item()

        train_losses.append(epoch_loss / len(train_loader))

        # Check validation accuracy after each epoch
        model.eval()
        val_correct, val_total = 0, 0
        with torch.no_grad():
            for features, labels in val_loader:
                features = features.to(DEVICE)
                outputs = model(features)
                preds = torch.argmax(outputs, dim=1)
                val_correct += (preds.cpu() == labels).sum().item()
                val_total += labels.size(0)

        epoch_val_acc = val_correct / val_total

        if epoch_val_acc > best_val_acc:
            best_val_acc = epoch_val_acc
            best_model_state = deepcopy(model.state_dict())
    
    train_time = time.time() - train_start
    gpu_mem_after = get_gpu_memory_mb()

    # Restore best model
    model.load_state_dict(best_model_state)
    
    # Final evaluation
    model.eval()
    with torch.no_grad():
        val_preds, val_true = [], []
        for features, labels in val_loader:
            features = features.to(DEVICE)
            outputs = model(features)
            preds = torch.argmax(outputs, dim=1).cpu().numpy()
            val_preds.extend(preds)
            val_true.extend(labels.numpy())
        
        # Test
        test_preds, test_true = [], []
        for features, labels in test_loader:
            features = features.to(DEVICE)
            outputs = model(features)
            preds = torch.argmax(outputs, dim=1).cpu().numpy()
            test_preds.extend(preds)
            test_true.extend(labels.numpy())
    
    val_acc = accuracy_score(val_true, val_preds)
    test_acc = accuracy_score(test_true, test_preds)
    
    metrics = {
        'val_acc': val_acc,
        'test_acc': test_acc,
        'test_f1_macro': f1_score(test_true, test_preds, average='macro', zero_division=0),
        'test_f1_weighted': f1_score(test_true, test_preds, average='weighted', zero_division=0),
        'test_precision_macro': precision_score(test_true, test_preds, average='macro', zero_division=0),
        'test_recall_macro': recall_score(test_true, test_preds, average='macro', zero_division=0),
        'test_preds': np.array(test_preds),
        'test_true': np.array(test_true),
        'val_preds': np.array(val_preds),
        'val_true': np.array(val_true),
        'train_time_sec': train_time,
        'gpu_mem_before_mb': gpu_mem_before,
        'gpu_mem_after_mb': gpu_mem_after,
        'gpu_mem_peak_mb': torch.cuda.max_memory_allocated() / (1024**2) if torch.cuda.is_available() else 0,
        'model_params': count_model_parameters(model),
        'model_size_mb': get_model_size_mb(model),
        'train_losses': train_losses,
    }
    return metrics

print("Enhanced evaluation function defined")


Enhanced evaluation function defined


## Feature Selection Methods
### 0. EvoloPy Metaheuristics

In [8]:
# ============================================================================
# EXACT EVOLOPY BAT IMPLEMENTATION FROM ORIGINAL CODE
# ============================================================================

def train_model_quick(model, train_loader, val_loader, epochs, lr, device):
    """Quick training for fitness evaluation"""
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    
    best_val_loss = float('inf')
    best_val_acc = 0.0
    
    for epoch in range(epochs):
        model.train()
        for features, labels in train_loader:
            features, labels = features.to(device), labels.to(device)
            optimizer.zero_grad()
            loss = criterion(model(features), labels)
            loss.backward()
            optimizer.step()
        
        model.eval()
        val_loss, val_correct, val_total = 0.0, 0, 0
        with torch.no_grad():
            for features, labels in val_loader:
                features, labels = features.to(device), labels.to(device)
                outputs = model(features)
                val_loss += criterion(outputs, labels).item()
                val_correct += (torch.argmax(outputs, 1) == labels).sum().item()
                val_total += labels.size(0)
        
        val_loss /= len(val_loader)
        val_acc = val_correct / val_total
        if val_loss < best_val_loss:
            best_val_loss, best_val_acc = val_loss, val_acc
    
    return best_val_loss, best_val_acc

# ============================================================================
# FITNESS FUNCTION: Using (1 - accuracy)
# ============================================================================
# Rationale: (1 - accuracy) is preferred over loss because:
#   - It directly optimizes the metric we care about (accuracy)
#   - It is bounded in [0, 1], making it cleaner for metaheuristic optimization
#   - Loss can vary in scale across different feature subsets
#   - Accuracy-based fitness is more interpretable and comparable across methods
#   - Less sensitive to calibration issues of the neural network

def create_fitness_function_evolopy(X_train, y_train, X_val, y_val, num_classes, total_features=None):
    """Create fitness function for EvoloPy using (1 - accuracy) + feature penalty"""
    eval_count = [0]  # track evaluations
    
    def fitness_function(binary_mask):
        try:
            if binary_mask.dtype != bool:
                binary_mask = binary_mask > 0.5

            num_selected = np.sum(binary_mask)
            if num_selected == 0:
                return 1.0
            
            X_tr_sel = X_train[:, binary_mask]
            X_val_sel = X_val[:, binary_mask]
            
            train_dataset = MultiModalDataset(X_tr_sel, y_train)
            val_dataset = MultiModalDataset(X_val_sel, y_val)
            train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
            val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE)
            
            model = SimpleNN(X_tr_sel.shape[1], num_classes).to(DEVICE)
            
            val_loss, val_acc = train_model_quick(
                model, train_loader, val_loader,
                epochs=30, lr=1e-3, device=DEVICE
            )
            
            del model, train_dataset, val_dataset, train_loader, val_loader
            torch.cuda.empty_cache()
            
            eval_count[0] += 1
            
            # Weighted fitness: accuracy + feature reduction pressure
            alpha = 0.95  # weight for accuracy
            beta = 0.05   # weight for feature reduction
            feature_ratio = num_selected / len(binary_mask)

            fitness = alpha * (1.0 - val_acc) + beta * feature_ratio
            return fitness
            
        except Exception as e:
            print(f"Error in fitness: {e}")
            return 1.0
        
    return fitness_function


def s_transfer(x):
    return 1 / (1 + np.exp(-10 * (x - 0.5)))


def run_evolopy_optimizer(optimizer_name, optimizer_func, X_train, y_train, X_val, y_val, num_classes, total_features=None, feature_dims=None):
    """Run any EvoloPy optimizer generically"""
    print(f"    Running EvoloPy {optimizer_name}...")
    print(f"      Fitness: (1 - accuracy), Pop: {N_POPULATION}, Iter: {MAX_ITERATIONS}")
    
    n_feats = total_features if total_features is not None else X_train.shape[1]
    fitness_func = create_fitness_function_evolopy(X_train, y_train, X_val, y_val, num_classes, n_feats)
    
    start = time.time()
    solution = optimizer_func(fitness_func, 0, 1, n_feats, N_POPULATION, MAX_ITERATIONS)
    exec_time = time.time() - start
    
    binary_mask = solution.bestIndividual > 0.5
    feat_dims = feature_dims if feature_dims is not None else FEATURE_DIMS
    modality_ret = calculate_modality_retention(binary_mask, feat_dims)
    
    results = {
        'mask': binary_mask,
        'convergence': solution.convergence.tolist() if hasattr(solution.convergence, 'tolist') else list(solution.convergence),
        'best_fitness': float(solution.convergence[-1]) if len(solution.convergence) > 0 else float('inf'),
        'execution_time': exec_time,
        'num_selected': int(np.sum(binary_mask)),
        'num_total': len(binary_mask),
        'modality_retention': modality_ret,
        'method': f'meta_{optimizer_name}'
    }
    
    print(f"      Selected: {results['num_selected']}/{results['num_total']} "
          f"({results['num_selected']/results['num_total']*100:.1f}%), "
          f"Fitness: {results['best_fitness']:.4f}, Time: {exec_time:.1f}s")
    print(f"      Modality: D={modality_ret['depth']['percentage']:.0f}%, "
          f"Sk={modality_ret['skeleton']['percentage']:.0f}%, ")
    
    return results

print("All EvoloPy metaheuristic runners defined")
print(f"Available optimizers: {list(EVOLOPY_OPTIMIZERS.keys())}")


All EvoloPy metaheuristic runners defined
Available optimizers: ['BAT', 'CS', 'DE', 'FFA', 'GA', 'GWO', 'HHO', 'JAYA', 'MFO', 'MVO', 'PSO', 'SCA', 'SSA', 'WOA']


### 1-3. Standard Feature Selection Methods (Mutual Info, RFE, LASSO)

In [9]:
def run_mutual_info_fs(X_train, y_train, X_val, y_val, num_classes, total_features=None, feature_dims=None):
    """Run Mutual Information feature selection"""
    print("    Running Mutual Information...")
    
    start_time = time.time()
    
    # Calculate mutual information scores
    mi_scores = mutual_info_classif(X_train, y_train, random_state=42)
    
    # Select top k features (target percentage)
    n_feats = total_features if total_features is not None else TOTAL_FEATURES
    k = int(n_feats * TARGET_FEATURE_PERCENTAGE)
    top_k_indices = np.argsort(mi_scores)[::-1][:k]
    
    binary_mask = np.zeros(n_feats, dtype=bool)
    binary_mask[top_k_indices] = True
    
    execution_time = time.time() - start_time
    
    feat_dims = feature_dims if feature_dims is not None else FEATURE_DIMS
    modality_retention = calculate_modality_retention(binary_mask, feat_dims)
    
    results = {
        'mask': binary_mask,
        'execution_time': execution_time,
        'num_selected': int(np.sum(binary_mask)),
        'num_total': len(binary_mask),
        'modality_retention': modality_retention,
        'mi_scores': mi_scores,
        'method': 'mutual_info'
    }
    
    print(f"      Selected: {results['num_selected']}/{results['num_total']} "
          f"({results['num_selected']/results['num_total']*100:.1f}%), "
          f"Time: {results['execution_time']:.1f}s")
    
    return results


def run_rfe_fs(X_train, y_train, X_val, y_val, num_classes, total_features=None, feature_dims=None):
    """Run RFE feature selection"""
    print("    Running RFE...")
    
    start_time = time.time()
    
    # Use Random Forest as base estimator
    estimator = RandomForestClassifier(n_estimators=50, random_state=42, n_jobs=-1)
    
    # Select top k features
    n_feats = total_features if total_features is not None else TOTAL_FEATURES
    k = int(n_feats * TARGET_FEATURE_PERCENTAGE)
    selector = RFE(estimator, n_features_to_select=k, step=50)  # Remove 50 features at a time
    selector.fit(X_train, y_train)
    
    binary_mask = selector.support_
    
    execution_time = time.time() - start_time
    
    feat_dims = feature_dims if feature_dims is not None else FEATURE_DIMS
    modality_retention = calculate_modality_retention(binary_mask, feat_dims)
    
    results = {
        'mask': binary_mask,
        'execution_time': execution_time,
        'num_selected': int(np.sum(binary_mask)),
        'num_total': len(binary_mask),
        'modality_retention': modality_retention,
        'ranking': selector.ranking_,
        'method': 'rfe'
    }
    
    print(f"      Selected: {results['num_selected']}/{results['num_total']} "
          f"({results['num_selected']/results['num_total']*100:.1f}%), "
          f"Time: {results['execution_time']:.1f}s")
    
    return results


def run_lasso_fs(X_train, y_train, X_val, y_val, num_classes, total_features=None, feature_dims=None):
    """Run LASSO feature selection"""
    print("    Running LASSO...")
    start_time = time.time()

    # Train Lasso to get feature importance scores
    lasso = LogisticRegression(
        penalty='l1',
        C=0.01,
        solver='saga',
        random_state=42,
        max_iter=1000
    )
    lasso.fit(X_train, y_train)

    # Rank features by coefficient magnitude across all classes
    coef_abs = np.abs(lasso.coef_).sum(axis=0)

    # Select top-k features to match target percentage
    n_feats = total_features if total_features is not None else TOTAL_FEATURES
    target_count = int(n_feats * TARGET_FEATURE_PERCENTAGE)
    top_indices = np.argsort(coef_abs)[::-1][:target_count]
    binary_mask = np.zeros(n_feats, dtype=bool)
    binary_mask[top_indices] = True

    execution_time = time.time() - start_time
    feat_dims = feature_dims if feature_dims is not None else FEATURE_DIMS
    modality_retention = calculate_modality_retention(binary_mask, feat_dims)

    results = {
        'mask': binary_mask,
        'execution_time': execution_time,
        'num_selected': int(np.sum(binary_mask)),
        'num_total': len(binary_mask),
        'modality_retention': modality_retention,
        'coefficients': coef_abs,
        'method': 'lasso'
    }

    print(f"      Selected: {results['num_selected']}/{results['num_total']} "
          f"({results['num_selected']/results['num_total']*100:.1f}%), "
          f"Time: {results['execution_time']:.1f}s")

    return results

print("Standard FS methods defined (Mutual Info, RFE, LASSO)")


Standard FS methods defined (Mutual Info, RFE, LASSO)


## Main Per-Fold Experiment Runner

In [40]:
def run_all_methods():
    """
    Run ALL methods with:
    1. Per-modality normalization (fit on train only)
    2. PCA on depth modality (fit on train only)
    3. Concatenation AFTER normalization+PCA
    Train on all subjects except one dedicated validation subject.
    """
    print("="*80)
    print("STARTING COMPREHENSIVE EXPERIMENTS (NO CROSS-VALIDATION)")
    print(f"Methods: baseline + 3 standard + {len(EVOLOPY_OPTIMIZERS)} metaheuristics = {len(ALL_METHODS)} total")
    print(f"Validation subject: {VAL_SUBJECT}")
    print(f"Per-modality normalization: ENABLED")
    print(f"Depth PCA: {'ENABLED -> ' + str(DEPTH_PCA_DIM) if DEPTH_PCA_DIM else 'DISABLED'}")
    print("="*80)

    num_classes = len(np.unique(y))

    # Master results dict: {method_name: result}
    master_results = {}

    # Build train/val indices based on dedicated validation subject
    val_subject_id = VAL_SUBJECT
    val_mask   = subjects == val_subject_id
    train_mask = ~val_mask

    train_idx = np.where(train_mask)[0]
    val_idx   = np.where(val_mask)[0]

    print(f"  Train: {len(train_idx)}, Val: {len(val_idx)}")
    print(f"  Train subjects: {sorted(np.unique(subjects[train_idx]).tolist())}")
    print(f"  Val subject: {np.unique(subjects[val_idx]).tolist()}")

    # =================================================================
    # Per-modality normalization + depth PCA  (fit on train only)
    # Pass val_idx as both val and test (no separate test set)
    # =================================================================
    X_train, X_val, X_val2, fold_feature_dims, pca_obj = prepare_fold_data_per_modality(
        X_per_modality, train_idx, val_idx, val_idx,
        depth_pca_dim=DEPTH_PCA_DIM
    )
    X_test = X_val  # no separate test split

    TOTAL_FEATURES = sum(fold_feature_dims.values())

    # Original dataset size
    orig_dataset_size_mb = get_dataset_size_mb(X_train) + get_dataset_size_mb(X_val)

    # =====================================================================
    # Run each method
    # =====================================================================
    for method_name in ALL_METHODS:
        print(f"\n  --- {method_name.upper()} ---")

        fs_start_time = time.time()

        # Feature selection
        if method_name == 'baseline':
            feature_mask = np.ones(TOTAL_FEATURES, dtype=bool)
            fs_results = {
                'mask': feature_mask,
                'execution_time': 0,
                'num_selected': TOTAL_FEATURES,
                'num_total': TOTAL_FEATURES,
                'method': 'baseline'
            }
        elif method_name == 'mutual_info':
            fs_results = run_mutual_info_fs(X_train, y[train_idx], X_val, y[val_idx], num_classes,
                                            total_features=TOTAL_FEATURES, feature_dims=fold_feature_dims)
            feature_mask = fs_results['mask']
        elif method_name == 'rfe':
            fs_results = run_rfe_fs(X_train, y[train_idx], X_val, y[val_idx], num_classes,
                                    total_features=TOTAL_FEATURES, feature_dims=fold_feature_dims)
            feature_mask = fs_results['mask']
        elif method_name == 'lasso':
            fs_results = run_lasso_fs(X_train, y[train_idx], X_val, y[val_idx], num_classes,
                                      total_features=TOTAL_FEATURES, feature_dims=fold_feature_dims)
            feature_mask = fs_results['mask']
        elif method_name.startswith('meta_'):
            opt_name = method_name.replace('meta_', '')
            opt_func = EVOLOPY_OPTIMIZERS[opt_name]
            fs_results = run_evolopy_optimizer(opt_name, opt_func,
                                               X_train, y[train_idx], X_val, y[val_idx], num_classes,
                                               total_features=TOTAL_FEATURES, feature_dims=fold_feature_dims)
            feature_mask = fs_results['mask']
        else:
            raise ValueError(f"Unknown method: {method_name}")

        fs_time = time.time() - fs_start_time

        # Apply feature mask
        X_train_sel = X_train[:, feature_mask]
        X_val_sel   = X_val[:, feature_mask]

        # Dataset size after selection
        opt_dataset_size_mb = get_dataset_size_mb(X_train_sel) + get_dataset_size_mb(X_val_sel)

        # Build and train final model
        model = SimpleNN(X_train_sel.shape[1], num_classes).to(DEVICE)

        train_dataset = MultiModalDataset(X_train_sel, y[train_idx])
        val_dataset   = MultiModalDataset(X_val_sel,   y[val_idx])

        train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
        val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE)

        if torch.cuda.is_available():
            torch.cuda.reset_peak_memory_stats()

        eval_metrics = train_and_evaluate_full(
            model, train_loader, val_loader, val_loader, N_EPOCHS, LEARNING_RATE, num_classes
        )

        # Modality retention
        if method_name != 'baseline':
            modality_ret = fs_results.get('modality_retention',
                calculate_modality_retention(feature_mask, fold_feature_dims))
        else:
            modality_ret = {m: {'selected': d, 'total': d, 'percentage': 100.0}
                           for m, d in fold_feature_dims.items()}

        # Compile result
        result = {
            'method': method_name,
            'val_subject': int(val_subject_id),
            'num_train': len(train_idx),
            'num_val': len(val_idx),
            # Accuracy & classification metrics
            'val_acc': eval_metrics['val_acc'],
            'val_f1_macro': eval_metrics['val_f1_macro'] if 'val_f1_macro' in eval_metrics else eval_metrics['test_f1_macro'],
            'val_f1_weighted': eval_metrics['val_f1_weighted'] if 'val_f1_weighted' in eval_metrics else eval_metrics['test_f1_weighted'],
            'val_precision_macro': eval_metrics['val_precision_macro'] if 'val_precision_macro' in eval_metrics else eval_metrics['test_precision_macro'],
            'val_recall_macro': eval_metrics['val_recall_macro'] if 'val_recall_macro' in eval_metrics else eval_metrics['test_recall_macro'],
            # Feature selection info
            'num_features_selected': int(np.sum(feature_mask)),
            'num_features_total': TOTAL_FEATURES,
            'feature_retention_pct': float(np.sum(feature_mask) / TOTAL_FEATURES * 100),
            'modality_retention': modality_ret,
            'feature_mask': feature_mask,
            # Feature dims (post-PCA)
            'feature_dims': fold_feature_dims,
            # Timing
            'fs_execution_time': fs_results.get('execution_time', 0),
            'train_time_sec': eval_metrics['train_time_sec'],
            'total_time_sec': fs_time + eval_metrics['train_time_sec'],
            # Model size
            'model_params': eval_metrics['model_params'],
            'model_size_mb': eval_metrics['model_size_mb'],
            # Dataset size
            'original_dataset_size_mb': orig_dataset_size_mb,
            'optimized_dataset_size_mb': opt_dataset_size_mb,
            'dataset_reduction_pct': float((1 - opt_dataset_size_mb / orig_dataset_size_mb) * 100) if orig_dataset_size_mb > 0 else 0,
            # GPU
            'gpu_mem_peak_mb': eval_metrics['gpu_mem_peak_mb'],
            # Convergence (metaheuristics only)
            'convergence': fs_results.get('convergence', None),
            'best_fitness': fs_results.get('best_fitness', None),
            # PCA info
            'depth_pca_dim': DEPTH_PCA_DIM,
            'depth_pca_variance_explained': float(pca_obj.explained_variance_ratio_.sum()) if (pca_obj is not None and hasattr(pca_obj, 'explained_variance_ratio_')) else None,
        }

        master_results[method_name] = result

        # Save mask and model
        method_dir = RESULTS_ROOT / method_name
        np.save(method_dir / "deployment_mask.npy", feature_mask)
        torch.save(model.state_dict(), method_dir / "deployment_model.pth")

        # Save result JSON (exclude large arrays)
        result_save = {k: v for k, v in result.items() if k not in ['feature_mask']}
        result_clean = {}
        for k, v in result_save.items():
            if isinstance(v, np.ndarray):
                result_clean[k] = v.tolist()
            elif isinstance(v, (np.floating, np.integer)):
                result_clean[k] = float(v)
            elif isinstance(v, dict):
                result_clean[k] = {}
                for kk, vv in v.items():
                    if isinstance(vv, dict):
                        result_clean[k][kk] = {kkk: float(vvv) if isinstance(vvv, (np.floating, np.integer)) else vvv for kkk, vvv in vv.items()}
                    else:
                        result_clean[k][kk] = float(vv) if isinstance(vv, (np.floating, np.integer)) else vv
            else:
                result_clean[k] = v

        with open(method_dir / "deployment_result.json", 'w') as f:
            json_lib.dump(result_clean, f, indent=2, default=str)

        print(
              f"Val Acc: {eval_metrics['val_acc']*100:.2f}%, "
              f"F1: {eval_metrics.get('val_f1_macro', eval_metrics['test_f1_macro'])*100:.2f}%, "
              f"Features: {np.sum(feature_mask)}/{TOTAL_FEATURES} "
              f"({np.sum(feature_mask)/TOTAL_FEATURES*100:.1f}%), "
              f"Model: {eval_metrics['model_size_mb']:.3f}MB")
        print(f"    Saved: deployment_mask.npy, deployment_model.pth, deployment_result.json")

        # Cleanup
        del model, train_dataset, val_dataset, train_loader, val_loader
        torch.cuda.empty_cache()
        gc.collect()

    print(f"\n{'='*80}")
    print("ALL EXPERIMENTS COMPLETED")
    print(f"{'='*80}")

    return master_results

print("Experiment runner defined (no cross-validation, single train/val split)")
print(f"  -> Dedicated validation subject: VAL_SUBJECT")
print(f"  -> Depth PCA: {'ENABLED -> ' + str(DEPTH_PCA_DIM) if DEPTH_PCA_DIM else 'DISABLED'}")


Experiment runner defined (no cross-validation, single train/val split)
  -> Dedicated validation subject: VAL_SUBJECT
  -> Depth PCA: ENABLED -> 512


## Run All Experiments

In [ ]:
master_results = run_all_methods()

STARTING COMPREHENSIVE EXPERIMENTS (NO CROSS-VALIDATION)
Methods: baseline + 3 standard + 14 metaheuristics = 18 total
Validation subject: 8
Per-modality normalization: ENABLED
Depth PCA: ENABLED -> 512
  Train: 754, Val: 107
  Train subjects: [1, 2, 3, 4, 5, 6, 7]
  Val subject: [8]
    KernelPCA on depth: 5508 -> 512
    Feature dims after processing: depth=512 | skeleton=1879 | TOTAL=2391

  --- BASELINE ---
Val Acc: 95.33%, F1: 95.15%, Features: 2391/2391 (100.0%), Model: 5.211MB
    Saved: deployment_mask.npy, deployment_model.pth, deployment_result.json

  --- MUTUAL_INFO ---
    Running Mutual Information...
      Selected: 1195/2391 (50.0%), Time: 40.1s
Val Acc: 94.39%, F1: 94.25%, Features: 1195/2391 (50.0%), Model: 2.875MB
    Saved: deployment_mask.npy, deployment_model.pth, deployment_result.json

  --- RFE ---
    Running RFE...
      Selected: 1195/2391 (50.0%), Time: 7.0s
Val Acc: 96.26%, F1: 96.33%, Features: 1195/2391 (50.0%), Model: 2.875MB
    Saved: deployment_mask.

## Results from SSA and WOA

In [10]:
EVOLOPY_OPTIMIZERS_SSA_WOA = {
    'SSA': SSA.SSA,
    'WOA': WOA.WOA,
}

ALL_METHODS_SSA_WOA = [f'meta_{k}' for k in EVOLOPY_OPTIMIZERS_SSA_WOA.keys()]

In [12]:
def run_all_methods_ssa_woa():
    print("="*80)
    print("STARTING COMPREHENSIVE EXPERIMENTS (NO CROSS-VALIDATION)")
    print(f"Methods: {len(EVOLOPY_OPTIMIZERS_SSA_WOA)} metaheuristics = {len(ALL_METHODS_SSA_WOA)} total")
    print(f"Validation subject: {VAL_SUBJECT}")
    print(f"Per-modality normalization: ENABLED")
    print(f"Depth PCA: {'ENABLED -> ' + str(DEPTH_PCA_DIM) if DEPTH_PCA_DIM else 'DISABLED'}")
    print("="*80)

    num_classes = len(np.unique(y))

    # Master results dict: {method_name: result}
    master_results = {}

    # Build train/val indices based on dedicated validation subject
    val_subject_id = VAL_SUBJECT
    val_mask   = subjects == val_subject_id
    train_mask = ~val_mask

    train_idx = np.where(train_mask)[0]
    val_idx   = np.where(val_mask)[0]

    print(f"  Train: {len(train_idx)}, Val: {len(val_idx)}")
    print(f"  Train subjects: {sorted(np.unique(subjects[train_idx]).tolist())}")
    print(f"  Val subject: {np.unique(subjects[val_idx]).tolist()}")

    # =================================================================
    # Per-modality normalization + depth PCA  (fit on train only)
    # Pass val_idx as both val and test (no separate test set)
    # =================================================================
    X_train, X_val, X_val2, fold_feature_dims, pca_obj = prepare_fold_data_per_modality(
        X_per_modality, train_idx, val_idx, val_idx,
        depth_pca_dim=DEPTH_PCA_DIM
    )
    X_test = X_val  # no separate test split

    TOTAL_FEATURES = sum(fold_feature_dims.values())

    # Original dataset size
    orig_dataset_size_mb = get_dataset_size_mb(X_train) + get_dataset_size_mb(X_val)

    # =====================================================================
    # Run each method
    # =====================================================================
    for method_name in ALL_METHODS_SSA_WOA:
        print(f"\n  --- {method_name.upper()} ---")

        fs_start_time = time.time()

        # Feature selection
        if method_name == 'baseline':
            feature_mask = np.ones(TOTAL_FEATURES, dtype=bool)
            fs_results = {
                'mask': feature_mask,
                'execution_time': 0,
                'num_selected': TOTAL_FEATURES,
                'num_total': TOTAL_FEATURES,
                'method': 'baseline'
            }
        elif method_name == 'mutual_info':
            fs_results = run_mutual_info_fs(X_train, y[train_idx], X_val, y[val_idx], num_classes,
                                            total_features=TOTAL_FEATURES, feature_dims=fold_feature_dims)
            feature_mask = fs_results['mask']
        elif method_name == 'rfe':
            fs_results = run_rfe_fs(X_train, y[train_idx], X_val, y[val_idx], num_classes,
                                    total_features=TOTAL_FEATURES, feature_dims=fold_feature_dims)
            feature_mask = fs_results['mask']
        elif method_name == 'lasso':
            fs_results = run_lasso_fs(X_train, y[train_idx], X_val, y[val_idx], num_classes,
                                      total_features=TOTAL_FEATURES, feature_dims=fold_feature_dims)
            feature_mask = fs_results['mask']
        elif method_name.startswith('meta_'):
            opt_name = method_name.replace('meta_', '')
            opt_func = EVOLOPY_OPTIMIZERS_SSA_WOA[opt_name]
            fs_results = run_evolopy_optimizer(opt_name, opt_func,
                                               X_train, y[train_idx], X_val, y[val_idx], num_classes,
                                               total_features=TOTAL_FEATURES, feature_dims=fold_feature_dims)
            feature_mask = fs_results['mask']
        else:
            raise ValueError(f"Unknown method: {method_name}")

        fs_time = time.time() - fs_start_time

        # Apply feature mask
        X_train_sel = X_train[:, feature_mask]
        X_val_sel   = X_val[:, feature_mask]

        # Dataset size after selection
        opt_dataset_size_mb = get_dataset_size_mb(X_train_sel) + get_dataset_size_mb(X_val_sel)

        # Build and train final model
        model = SimpleNN(X_train_sel.shape[1], num_classes).to(DEVICE)

        train_dataset = MultiModalDataset(X_train_sel, y[train_idx])
        val_dataset   = MultiModalDataset(X_val_sel,   y[val_idx])

        train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
        val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE)

        if torch.cuda.is_available():
            torch.cuda.reset_peak_memory_stats()

        eval_metrics = train_and_evaluate_full(
            model, train_loader, val_loader, val_loader, N_EPOCHS, LEARNING_RATE, num_classes
        )

        # Modality retention
        if method_name != 'baseline':
            modality_ret = fs_results.get('modality_retention',
                calculate_modality_retention(feature_mask, fold_feature_dims))
        else:
            modality_ret = {m: {'selected': d, 'total': d, 'percentage': 100.0}
                           for m, d in fold_feature_dims.items()}

        # Compile result
        result = {
            'method': method_name,
            'val_subject': int(val_subject_id),
            'num_train': len(train_idx),
            'num_val': len(val_idx),
            # Accuracy & classification metrics
            'val_acc': eval_metrics['val_acc'],
            'val_f1_macro': eval_metrics['val_f1_macro'] if 'val_f1_macro' in eval_metrics else eval_metrics['test_f1_macro'],
            'val_f1_weighted': eval_metrics['val_f1_weighted'] if 'val_f1_weighted' in eval_metrics else eval_metrics['test_f1_weighted'],
            'val_precision_macro': eval_metrics['val_precision_macro'] if 'val_precision_macro' in eval_metrics else eval_metrics['test_precision_macro'],
            'val_recall_macro': eval_metrics['val_recall_macro'] if 'val_recall_macro' in eval_metrics else eval_metrics['test_recall_macro'],
            # Feature selection info
            'num_features_selected': int(np.sum(feature_mask)),
            'num_features_total': TOTAL_FEATURES,
            'feature_retention_pct': float(np.sum(feature_mask) / TOTAL_FEATURES * 100),
            'modality_retention': modality_ret,
            'feature_mask': feature_mask,
            # Feature dims (post-PCA)
            'feature_dims': fold_feature_dims,
            # Timing
            'fs_execution_time': fs_results.get('execution_time', 0),
            'train_time_sec': eval_metrics['train_time_sec'],
            'total_time_sec': fs_time + eval_metrics['train_time_sec'],
            # Model size
            'model_params': eval_metrics['model_params'],
            'model_size_mb': eval_metrics['model_size_mb'],
            # Dataset size
            'original_dataset_size_mb': orig_dataset_size_mb,
            'optimized_dataset_size_mb': opt_dataset_size_mb,
            'dataset_reduction_pct': float((1 - opt_dataset_size_mb / orig_dataset_size_mb) * 100) if orig_dataset_size_mb > 0 else 0,
            # GPU
            'gpu_mem_peak_mb': eval_metrics['gpu_mem_peak_mb'],
            # Convergence (metaheuristics only)
            'convergence': fs_results.get('convergence', None),
            'best_fitness': fs_results.get('best_fitness', None),
            # PCA info
            'depth_pca_dim': DEPTH_PCA_DIM,
            'depth_pca_variance_explained': float(pca_obj.explained_variance_ratio_.sum()) if (pca_obj is not None and hasattr(pca_obj, 'explained_variance_ratio_')) else None,
        }

        master_results[method_name] = result

        # Save mask and model
        method_dir = RESULTS_ROOT / method_name
        np.save(method_dir / "deployment_mask.npy", feature_mask)
        torch.save(model.state_dict(), method_dir / "deployment_model.pth")

        # Save result JSON (exclude large arrays)
        result_save = {k: v for k, v in result.items() if k not in ['feature_mask']}
        result_clean = {}
        for k, v in result_save.items():
            if isinstance(v, np.ndarray):
                result_clean[k] = v.tolist()
            elif isinstance(v, (np.floating, np.integer)):
                result_clean[k] = float(v)
            elif isinstance(v, dict):
                result_clean[k] = {}
                for kk, vv in v.items():
                    if isinstance(vv, dict):
                        result_clean[k][kk] = {kkk: float(vvv) if isinstance(vvv, (np.floating, np.integer)) else vvv for kkk, vvv in vv.items()}
                    else:
                        result_clean[k][kk] = float(vv) if isinstance(vv, (np.floating, np.integer)) else vv
            else:
                result_clean[k] = v

        with open(method_dir / "deployment_result.json", 'w') as f:
            json_lib.dump(result_clean, f, indent=2, default=str)

        print(
              f"Val Acc: {eval_metrics['val_acc']*100:.2f}%, "
              f"F1: {eval_metrics.get('val_f1_macro', eval_metrics['test_f1_macro'])*100:.2f}%, "
              f"Features: {np.sum(feature_mask)}/{TOTAL_FEATURES} "
              f"({np.sum(feature_mask)/TOTAL_FEATURES*100:.1f}%), "
              f"Model: {eval_metrics['model_size_mb']:.3f}MB")
        print(f"    Saved: deployment_mask.npy, deployment_model.pth, deployment_result.json")

        # Cleanup
        del model, train_dataset, val_dataset, train_loader, val_loader
        torch.cuda.empty_cache()
        gc.collect()

    print(f"\n{'='*80}")
    print("ALL EXPERIMENTS COMPLETED")
    print(f"{'='*80}")

    return master_results

print("Experiment runner defined (no cross-validation, single train/val split)")
print(f"  -> Dedicated validation subject: VAL_SUBJECT")
print(f"  -> Depth PCA: {'ENABLED -> ' + str(DEPTH_PCA_DIM) if DEPTH_PCA_DIM else 'DISABLED'}")


Experiment runner defined (no cross-validation, single train/val split)
  -> Dedicated validation subject: VAL_SUBJECT
  -> Depth PCA: ENABLED -> 512


In [13]:
master_results_ssa_woa = run_all_methods_ssa_woa()

STARTING COMPREHENSIVE EXPERIMENTS (NO CROSS-VALIDATION)
Methods: 2 metaheuristics = 2 total
Validation subject: 8
Per-modality normalization: ENABLED
Depth PCA: ENABLED -> 512
  Train: 754, Val: 107
  Train subjects: [1, 2, 3, 4, 5, 6, 7]
  Val subject: [8]
    KernelPCA on depth: 5508 -> 512
    Feature dims after processing: depth=512 | skeleton=1879 | TOTAL=2391

  --- META_SSA ---
    Running EvoloPy SSA...
      Fitness: (1 - accuracy), Pop: 20, Iter: 30
SSA is optimizing  "fitness_function"
['At iteration 1 the best fitness is 0.06946571449790297']
['At iteration 2 the best fitness is 0.06946571449790297']
['At iteration 3 the best fitness is 0.06946571449790297']
['At iteration 4 the best fitness is 0.06946571449790297']
['At iteration 5 the best fitness is 0.06946571449790297']
['At iteration 6 the best fitness is 0.06946571449790297']
['At iteration 7 the best fitness is 0.06087997435867369']
['At iteration 8 the best fitness is 0.06087997435867369']
['At iteration 9 the best

## Compile results together

In [14]:
# RELOAD ALL FOLD RESULTS FROM DISK

import json as json_lib
from pathlib import Path

master_results = {}

for method_name in ALL_METHODS:
    fold_dir = RESULTS_ROOT / method_name
    json_path = fold_dir / f"deployment_result.json"
    mask_path = fold_dir / f"deployment_mask.npy"
        
    if json_path.exists():
        with open(json_path, 'r') as f:
            fold_data = json_lib.load(f)
            
        # Reload the binary mask from .npy (not stored in JSON)
        if mask_path.exists():
            fold_data['feature_mask'] = np.load(mask_path)
            
        master_results[method_name] = fold_data
        print(f"  Loaded: {method_name}")
    else:
        print(f"  MISSING: {method_name}")

  Loaded: baseline
  Loaded: mutual_info
  Loaded: rfe
  Loaded: lasso
  Loaded: meta_BAT
  Loaded: meta_CS
  Loaded: meta_DE
  Loaded: meta_FFA
  Loaded: meta_GA
  Loaded: meta_GWO
  Loaded: meta_HHO
  Loaded: meta_JAYA
  Loaded: meta_MFO
  Loaded: meta_MVO
  Loaded: meta_PSO
  Loaded: meta_SCA
  Loaded: meta_SSA
  Loaded: meta_WOA


## Build Comprehensive Per-Fold Results CSV

In [19]:
# Build a DataFrame with one row per method (no folds)
FEATURE_DIMS = None
TOTAL_FEATURES = None

rows = []

for method_name in ALL_METHODS:
    if method_name not in master_results:
        continue
    r = master_results[method_name]

    # Extract feature dims from first available result
    if FEATURE_DIMS is None and 'feature_dims' in r:
        FEATURE_DIMS = r['feature_dims']
        TOTAL_FEATURES = sum(FEATURE_DIMS.values())
        print(f"Post-PCA feature dims: {FEATURE_DIMS}, Total: {TOTAL_FEATURES}")

    row = {
        'Method': method_name,
        'Val Accuracy (%)': r['val_acc'] * 100,
        'F1 Macro (%)': r['val_f1_macro'] * 100,
        'F1 Weighted (%)': r['val_f1_weighted'] * 100,
        'Precision Macro (%)': r['val_precision_macro'] * 100,
        'Recall Macro (%)': r['val_recall_macro'] * 100,
        'Features Selected': r['num_features_selected'],
        'Features Total': r['num_features_total'],
        'Feature Retention (%)': r['feature_retention_pct'],
        'FS Time (s)': r['fs_execution_time'],
        'Train Time (s)': r['train_time_sec'],
        'Total Time (s)': r['total_time_sec'],
        'Model Params': r['model_params'],
        'Model Size (MB)': r['model_size_mb'],
        'Orig Dataset (MB)': r['original_dataset_size_mb'],
        'Opt Dataset (MB)': r['optimized_dataset_size_mb'],
        'Dataset Reduction (%)': r['dataset_reduction_pct'],
        'GPU Peak (MB)': r['gpu_mem_peak_mb'],
        'N Train': r['num_train'],
        'N Val': r['num_val'],
        'Val Subject': r['val_subject'],
    }

    # Per-modality retention
    for mod in MODALITY_NAMES:
        if mod in r.get('modality_retention', {}):
            row[f'{mod}_retained'] = r['modality_retention'][mod]['selected']
            row[f'{mod}_total'] = r['modality_retention'][mod]['total']
            row[f'{mod}_retention_%'] = r['modality_retention'][mod]['percentage']

    rows.append(row)

df_all = pd.DataFrame(rows)
df_all.to_csv(RESULTS_ROOT / "all_results.csv", index=False)
print(f"Results table: {df_all.shape}")
print(f"Methods: {df_all['Method'].nunique()}")
df_all


Post-PCA feature dims: {'depth': 512, 'skeleton': 1879}, Total: 2391
Results table: (18, 27)
Methods: 18


,Method,Val Accuracy (%),F1 Macro (%),F1 Weighted (%),Precision Macro (%),Recall Macro (%),Features Selected,Features Total,Feature Retention (%),FS Time (s),...,GPU Peak (MB),N Train,N Val,Val Subject,depth_retained,depth_total,depth_retention_%,skeleton_retained,skeleton_total,skeleton_retention_%
0,baseline,95.327103,95.149912,95.104584,95.925926,95.370370,2391,2391,100.000000,0.000000,...,116.216309,754,107,8,512,512,100.000000,1879,1879,100.000000
1,mutual_info,94.392523,94.250441,94.196707,96.296296,94.444444,1195,2391,49.979088,40.072419,...,45.874512,754,107,8,12,512,2.343750,1183,1879,62.959021
2,rfe,96.261682,96.334509,96.300252,97.098765,96.296296,1195,2391,49.979088,6.988390,...,45.874512,754,107,8,120,512,23.437500,1075,1879,57.211283
3,lasso,92.523364,91.075838,90.992434,92.592593,92.592593,1195,2391,49.979088,245.941644,...,45.874512,754,107,8,15,512,2.929688,1180,1879,62.799361
4,meta_BAT,95.327103,94.886430,94.838640,96.931217,95.370370,1132,2391,47.344207,1570.001068,...,45.251465,754,107,8,241,512,47.070312,891,1879,47.418840
5,meta_CS,96.261682,96.178718,96.143005,97.037037,96.296296,1147,2391,47.971560,3558.958989,...,45.399902,754,107,8,238,512,46.484375,909,1879,48.376796
6,meta_DE,99.065421,99.059377,99.050586,99.259259,99.074074,1159,2391,48.473442,1802.338145,...,45.518555,754,107,8,270,512,52.734375,889,1879,47.312400
7,meta_FFA,98.130841,98.118754,98.101172,98.518519,98.148148,1173,2391,49.058971,1210.015293,...,45.657227,754,107,8,240,512,46.875000,933,1879,49.654071
8,meta_GA,98.130841,98.133451,98.116007,98.333333,98.148148,1204,2391,50.355500,1151.733679,...,45.963379,754,107,8,248,512,48.437500,956,1879,50.878127
9,meta_GWO,99.065421,99.059377,99.050586,99.259259,99.074074,873,2391,36.511920,1123.413811,...,41.429199,754,107,8,198,512,38.671875,675,1879,35.923363
